4. Tải gói `modal_upload.zip` từ dự án lên `/mnt/agqo/input/` (hoặc tải riêng hai CSV `ohlcv_daily.csv` và `members.csv`; tệp `manifest.json` là tùy chọn).
5. Chạy lần lượt các ô từ trên xuống. Các tệp đầu ra sẽ nằm trong `/mnt/agqo/results/` và có thể tải xuống từ bảng tệp.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_URL = "https://github.com/jncmtam/UTEL_AQ2026.git"
PROJECT_DIR = Path("/mnt/agqo/project")
PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "main", PROJECT_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
print(f"Mã dự án: {PROJECT_DIR}")

In [ ]:
# Modal Notebook có sẵn PyTorch. Cài gói dự án và thư viện Qwen3 còn thiếu.
%pip install -e /mnt/agqo/project 'transformers>=4.51,<5' 'accelerate>=1.6' 

In [ ]:
import json
import zipfile
import pandas as pd
import torch
from IPython.display import display

DATA_DIR = Path("/mnt/agqo/input")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_ARCHIVE = DATA_DIR / "modal_upload.zip"
OHLCV_PATH = DATA_DIR / "ohlcv_daily.csv"
MEMBERS_PATH = DATA_DIR / "members.csv"
MANIFEST_PATH = DATA_DIR / "manifest.json"

if DATA_ARCHIVE.exists() and not (OHLCV_PATH.exists() and MEMBERS_PATH.exists()):
    with zipfile.ZipFile(DATA_ARCHIVE) as bundle:
        allowed = {"ohlcv_daily.csv", "members.csv", "manifest.json"}
        names = {Path(name).name for name in bundle.namelist()}
        if not {"ohlcv_daily.csv", "members.csv"}.issubset(names):
            raise ValueError("Gói dữ liệu cần chứa ohlcv_daily.csv và members.csv.")
        for name in bundle.namelist():
            if Path(name).name in allowed and not name.endswith("/"):
                (DATA_DIR / Path(name).name).write_bytes(bundle.read(name))

missing = [str(path) for path in (OHLCV_PATH, MEMBERS_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Hãy tải modal_upload.zip hoặc các tệp CSV lên /mnt/agqo/input/ bằng bảng tệp Modal rồi chạy lại ô này: "
        + ", ".join(missing)
    )

bars = pd.read_csv(OHLCV_PATH)
members = pd.read_csv(MEMBERS_PATH)
print(f"Dữ liệu: {len(bars):,} dòng; {bars['ticker'].nunique()} mã; "
      f"{bars['date'].min()} đến {bars['date'].max()}")
print(f"Danh sách VN30: {len(members)} mã")
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print("Bản kê khai:", manifest.get("dataset_name"), manifest.get("retrieved_at_utc"))
display(bars.head())
display(members.head())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có")

## Chọn và nạp Qwen3

Mặc định dùng `Qwen/Qwen3-4B` để đọc tiếng Việt và đối chiếu số liệu. Mô hình được chạy trực tiếp bằng Transformers trên GPU Modal, không qua API. T4/L4 phù hợp hơn cho cấu hình này. Nếu chỉ có CPU, đổi thành `Qwen/Qwen3-0.6B` và đặt `REQUIRE_GPU = False`; chất lượng suy luận có thể thấp hơn và tốc độ chậm hơn.

Lời nhắc chỉ gửi thống kê tổng hợp theo từng thời điểm, danh mục chính sách và chỉ số hiệu quả đã khép. Mã cổ phiếu và nến OHLCV thô không được đưa vào lời nhắc. Câu trả lời phải qua bộ kiểm tra lược đồ và danh sách chính sách cho phép trước khi áp dụng.

In [ ]:
# Lưu trọng số trên Volume để lần khởi động sau không phải tải lại.
os.environ["HF_HOME"] = "/mnt/agqo/hf-cache"

MODEL_NAME = "Qwen/Qwen3-4B"
REQUIRE_GPU = True
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn GPU T4/L4 trong cấu hình Modal Notebook rồi khởi động lại nhân.")
print(f"Sẵn sàng tải {MODEL_NAME}; bộ nhớ đệm: {os.environ['HF_HOME']}")
print("Mô hình sẽ được nạp đúng một lần trong tiến trình chạy walk-forward.")

## Chạy walk-forward và QAOA

Lệnh này chạy lại thí nghiệm đầy đủ đã có trong dự án. Ở mỗi tháng kiểm tra, AGQO dùng thông tin đã có đến ngày tín hiệu; Qwen3 không được thấy lợi nhuận của kỳ đang chọn. Danh mục đối chứng chính sách cố định vẫn được mô phỏng để so sánh. Do mỗi chính sách được kiểm định qua nhiều hạt giống và nhiều lần tối ưu, lần chạy đầy đủ có thể mất thời gian.

In [ ]:
OUTPUT_DIR = Path("/mnt/agqo/results")
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULT_JSON = OUTPUT_DIR / "vn30_ai_qaoa_walkforward.json"
REPORT_MD = OUTPUT_DIR / "BAO_CAO_AI_QAOA_WALK_FORWARD.md"
GUIDE_CACHE = OUTPUT_DIR / "agqo_qwen3_decisions.json"

command = [
    sys.executable,
    str(PROJECT_DIR / "scripts/run_ai_qaoa_walkforward.py"),
    "--data", str(OHLCV_PATH),
    "--members", str(MEMBERS_PATH),
    "--json", str(RESULT_JSON),
    "--report", str(REPORT_MD),
    "--figures", str(FIGURES_DIR),
    "--guide-cache", str(GUIDE_CACHE),
    "--llm-guide", "qwen3",
    "--qwen3-model", MODEL_NAME,
]
print("Đang chạy:", " ".join(command))
env = os.environ.copy()
env["HF_HOME"] = "/mnt/agqo/hf-cache"
subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)

## Xem kết quả và insight

In [ ]:
result = json.loads(RESULT_JSON.read_text(encoding="utf-8"))
performance = result["portfolio"]["performance"]
selected_names = [
    name for name in performance
    if name == "AGQO LLM→QAOA / chính sách thích nghi / 12 mã thanh khoản cao"
    or name == "VN30"
    or name == "VN-Index"
    or "Ledoit–Wolf" in name
    or "QAOA X, p=1" in name
]
rows = []
for name in selected_names:
    metrics = performance[name]["metrics"]
    rows.append({
        "Phương pháp": name,
        "CAGR ròng": metrics.get("cagr"),
        "Sharpe": metrics.get("sharpe"),
        "Sortino": metrics.get("sortino"),
        "Sụt giảm tối đa": metrics.get("max_drawdown"),
        "Hệ số lợi nhuận": metrics.get("profit_factor"),
        "Số kỳ": metrics.get("periods"),
    })
display(pd.DataFrame(rows).set_index("Phương pháp").style.format(precision=3))
print("Số lần Qwen3 chọn cấu hình:", result["agqo_guidance"]["decision_count"])
print("Mô hình:", MODEL_NAME)
print("Phiên bản trọng số:", sorted({d.get("model_revision") for d in result["agqo_guidance"]["decisions"] if d.get("model_revision")}))
print("Dùng bộ nhớ đệm quyết định:", GUIDE_CACHE)

In [ ]:
from IPython.display import Image, display

figure_paths = sorted(FIGURES_DIR.glob("*.png"))
print(f"Có {len(figure_paths)} biểu đồ")
for path in figure_paths:
    print(path.name)
    display(Image(filename=str(path)))

## Tải kết quả về máy

Tệp kết quả chi tiết, báo cáo tiếng Việt, nhật ký quyết định Qwen3 và biểu đồ nằm trong `/mnt/agqo/results/`. Tải từng tệp từ bảng tệp của Modal. Ô tiếp theo đóng gói đầu ra nghiên cứu để tải một lần; dữ liệu đầu vào OHLCV không được đưa vào gói.

In [ ]:
import zipfile

ARCHIVE_PATH = OUTPUT_DIR / "agqo_qwen3_results.zip"
with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (RESULT_JSON, REPORT_MD, GUIDE_CACHE):
        if path.exists():
            archive.write(path, arcname=path.name)
    for path in figure_paths:
        archive.write(path, arcname=f"figures/{path.name}")
print(f"Gói kết quả: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1024 / 1024:.1f} MiB)")
print("Mở bảng Files của Modal, vào /mnt/agqo/results/ và tải agqo_qwen3_results.zip")